## Módulo 4 - Sistema de Recomendación Híbrido + Chatbot (Recomendador)

### Introducción

En este módulo construimos la capa final del sistema: un recomendador híbrido conversacional.

Los módulos anteriores producen dos fuentes de evidencia complementarias:

1. El módulo semántico recupera hoteles a partir de la similitud entre la consulta del usuario y las reviews vectorizadas. Este motor es especialmente útil para capturar necesidades abiertas o subjetivas, como “tranquilo”, “linda vista”, “romántico”, “cómodo para descansar” o “ideal para ir en familia”.

2. El módulo estructurado utiliza los atributos extraídos previamente desde las reviews, como wifi, vista, ubicación, ruido, limpieza, desayuno, habitación, seguridad, entre otros. Este motor permite medir fortalezas concretas de cada hotel a nivel de categoría.

A diferencia de una integración basada únicamente en un agente que llama herramientas separadas, en este módulo construimos una capa explícita de ranking híbrido. El sistema genera dos listas de candidatos: una lista semántica y una lista estructurada. Luego normaliza los scores de cada lista, identifica qué hoteles aparecen en ambas, y prioriza aquellos que tienen doble evidencia.

La respuesta final se organiza en dos bloques:

- Recomendaciones principales: hoteles que aparecen bien posicionados tanto en la búsqueda semántica como en la búsqueda por atributos.
- Otras opciones que también podrían interesarte: hoteles fuertes en una sola de las dos señales, ya sea por similitud semántica o por atributos estructurados.

De esta forma, el agente no combina los motores solamente al redactar la respuesta, sino que se apoya en una función determinística de re-ranking híbrido.

### Arquitectura

El chatbot se implementa como una interfaz conversacional sobre un sistema de recomendación híbrido. La arquitectura combina tres componentes principales:

- **LLM (GPT-4o-mini):** interpreta el lenguaje natural del usuario, identifica el destino, separa la intención semántica de los atributos estructurados relevantes y redacta la respuesta final en un tono conversacional.

- **Ranker híbrido:** función central del módulo. Este componente combina dos fuentes de evidencia: una lista semántica, generada a partir de embeddings y búsqueda vectorial sobre ChromaDB, y una lista estructurada, generada a partir de los atributos extraídos en el módulo 2 y almacenados en SQLite. Ambas listas se normalizan, se cruzan por hotel y se reorganizan para priorizar los hoteles que aparecen bien posicionados en ambos rankings.

- **Tools del agente:** funciones de Python que el LLM puede invocar para obtener información del sistema. En lugar de exponer dos motores separados como salida final, el agente utiliza una herramienta principal de recomendación híbrida, que devuelve recomendaciones principales y opciones alternativas.

Es importante notar que el LLM no realiza la búsqueda ni calcula los scores directamente. Su rol es interpretar la consulta del usuario y traducirla a una llamada concreta al sistema: destino, consulta semántica y atributos relevantes. El trabajo pesado sigue estando en los componentes desarrollados previamente: el motor semántico del módulo 3, el motor estructurado del módulo 2 y la nueva capa de re-ranking híbrido que los combina.

La integración entre motores no queda delegada únicamente a la redacción del agente. En este módulo, la combinación ocurre antes de la respuesta final, mediante una lógica explícita que cruza los rankings semántico y estructurado. Por eso, las recomendaciones principales corresponden a hoteles con doble evidencia: aparecen como relevantes tanto por similitud semántica como por atributos concretos.

### Por Qué LangChain

Elegimos LangChain como framework de orquestación porque ofrece una abstracción estándar y bien documentada para construir agentes con tool calling, sin tener que implementar el loop de razonamiento manualmente. Para este MVP utilizamos un subconjunto mínimo: definimos las tools como funciones Python decoradas y armamos un agente con `create_tool_calling_agent`. No se utilizan abstracciones más complejas como chains anidadas o memoria persistente, lo cual mantiene el código simple y transparente.

### Por Qué GPT-4o-mini

Para esta primera versión usamos GPT-4o-mini de OpenAI. Es uno de los modelos más económicos disponibles, con calidad más que suficiente para la tarea (parsear una consulta corta y redactar una respuesta a partir de resultados estructurados). Cada interacción del chatbot cuesta una fracción mínima en términos de tokens, lo que hace viable correrlo extensivamente durante la demo y la presentación.

### Set-Up Inicial

Instalamos las librerías necesarias: langchain (orquestación del agente), langchain-openai (conector con GPT-4o-mini), chromadb (para conectar a la base de datos persistente), y sentence-transformers (para vectorizar la consulta del usuario).

In [1]:
%pip install -q langchain langchain-openai langchain-chroma chromadb sentence-transformers


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Fijamos la versión de LangChain a 0.3.27 porque es la versión que tiene mejor documentación y estabilidad al momento del trabajo. Las versiones más recientes (1.x) tienen una API distinta que no se condice con la mayoría de los tutoriales disponibles.

In [2]:
# Esta versión de LangChain es un poco mas vieja pero está mejor documentada en internet
%pip install -U "langchain==0.3.27" "langchain-openai==0.3.34" "langchain-core==0.3.78"


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Importamos todas las librerías que vamos a usar a lo largo del notebook.

In [3]:
# Importamos librerías
from sentence_transformers import SentenceTransformer
import chromadb
import pandas as pd
import numpy as np
import os
import json
from pathlib import Path
import sqlite3
import yaml

### Configuración Del LLM

Cargamos el LLM antes que los motores de recomendación, porque el motor estructurado depende del LLM para generar queries SQL dinámicamente a partir de las consultas del usuario. El motor semántico no lo necesita, pero por orden y prolijidad cargamos todas las piezas centrales antes de armar la lógica que las usa.

Cargamos la API key de OpenAI como variable de entorno. Esto evita escribirla directamente en el código (buena práctica de seguridad, especialmente si el notebook se sube a GitHub). En Colab usamos el sistema de Secrets integrado: la key se carga una sola vez en la configuración del notebook y se accede vía userdata.get(...). En entornos locales (VSCode, scripts), se suele usar un archivo .env con la librería python-dotenv.

In [4]:
# Importamos librería
from dotenv import load_dotenv

load_dotenv()

# Cargamos la API key
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Validamos
assert OPENAI_API_KEY is not None, "Falta OPENAI_API_KEY en el archivo .env"

Ahora si, cargamos el LLM y corremos un test mínimo para garantizar funcionamiento.

In [5]:
# Importo librería
from langchain_openai import ChatOpenAI

# temperature = 0 → respuestas más deterministas (útil para que el agente sea consistente)
llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0)

# Test mínimo: que responda algo trivial
respuesta = llm.invoke("Decí 'Hola, todo funciona' y nada más.")
print(respuesta.content)

Hola, todo funciona.


### Motor Estructurado (Módulo 2)

En esta sección preparamos el motor estructurado construido a partir del módulo 2. A diferencia del motor semántico, que opera sobre embeddings de texto libre, este motor opera sobre una tabla SQLite donde cada hotel tiene un score numérico por categoría predefinida, como wifi, vista, ubicación, ruido, limpieza, desayuno, habitación, seguridad, entre otras.

En el nuevo enfoque híbrido, este motor no se usa como una recomendación final independiente, sino como el generador de la Lista B: un ranking de hoteles basado en atributos estructurados.

La lógica general se mantiene igual que en el módulo 2:

1. Nos conectamos a la base SQLite generada (`hoteles.db`).

2. Cargamos el YAML de categorías para saber qué atributos existen y cómo interpretarlos.

3. Usamos `generar_query_sql` para traducir la consulta del usuario a una query SQL válida. Esta query permite recuperar hoteles candidatos según los atributos concretos mencionados por el usuario.

4. Ejecutamos la query contra SQLite.

5. Aplicamos `score_hotel` para ordenar los hoteles candidatos según sus atributos positivos normalizados.

La diferencia principal está en el uso posterior de esta salida. Antes, el ranking estructurado podía presentarse directamente al usuario. Ahora, se normaliza y se cruza con el ranking semántico. Los hoteles que aparecen en ambas listas se priorizan como recomendaciones principales, mientras que los hoteles fuertes solo por atributos pueden conservarse como alternativas estructuradas.

A continuación cargamos el output del módulo 2 (la tabla SQL con un score agregado por categoría a nivel hotel). Esta tabla nos va a permitir aplicar filtros duros por atributos sobre los hoteles.

In [6]:
# Cargamos la base de datos tabular
RUTA_HOTELES_DB = Path.cwd().parent / "Data" / "hoteles.db"
conn_hoteles = sqlite3.connect(str(RUTA_HOTELES_DB))

In [7]:
# Vemos cuántos hoteles hay y qué columnas tiene
test = pd.read_sql_query("SELECT COUNT(*) as total FROM hoteles", conn_hoteles)
print(f"Hoteles en la base: {test['total'].iloc[0]:,}")

# Vemos las primeras filas y las columnas disponibles
test_head = pd.read_sql_query("SELECT * FROM hoteles LIMIT 3", conn_hoteles)
print(f"\nColumnas: {list(test_head.columns)}")
test_head

Hoteles en la base: 2,523

Columnas: ['hotel_id_review', 'wifi', 'vista', 'ubicacion', 'ruido', 'limpieza', 'transporte', 'personal', 'desayuno', 'restaurante', 'habitacion', 'aire_acondicionado', 'estacionamiento', 'seguridad', 'checkin', 'destino']


,hotel_id_review,wifi,vista,ubicacion,ruido,limpieza,transporte,personal,desayuno,restaurante,habitacion,aire_acondicionado,estacionamiento,seguridad,checkin,destino
0,200031,0,0,10,-1,-7,0,3,-1,-3,-7,-1,0,-1,0,México - Guadalajara
1,200151,-1,0,8,-1,-5,-2,9,-1,0,2,-1,-2,1,0,México - Guadalajara
2,200209,-1,0,9,0,-11,-1,-2,-5,-5,-4,0,-3,0,-2,México - Guadalajara


Luego, cargamos el archivo config_features.yaml, el mismo que define las categorías de atributos del módulo 2. Lo traemos al chatbot por dos motivos. El primero es para que el agente conozca qué categorías existen y pueda mapear correctamente las menciones del usuario (por ejemplo, "el aire andaba mal" refiere a aire_acondicionado). El segundo es para mantener el sistema sincronizado con el yaml. Si en una próxima iteración agregamos o modificamos categorías, el chatbot las incorpora automáticamente sin tocar código.

In [8]:
# Cargamos el archivo YAML con la configuración de categorías
RUTA_FEATURE_CONFIG = Path.cwd() / "config_features.yaml"
with open(RUTA_FEATURE_CONFIG, "r", encoding = "utf-8") as f:
    config_categorias = yaml.safe_load(f)

categorias = config_categorias["categorias"]
print(f"Categorías cargadas: {[c['nombre'] for c in categorias]}")

Categorías cargadas: ['wifi', 'vista', 'ubicacion', 'ruido', 'limpieza', 'transporte', 'personal', 'desayuno', 'restaurante', 'habitacion', 'aire_acondicionado', 'estacionamiento', 'seguridad', 'checkin']


Para complementar la búsqueda semántica con la información estructurada del módulo 2, definimos la función score_hotel. Esta función recibe un conjunto de hoteles candidatos (típicamente el resultado de una consulta SQL sobre la base de atributos) y los rankea según un score agregado por categoría.

El procedimiento consta de tres pasos:

1. Primero, llevamos a cero los valores negativos: los scores negativos representan menciones desfavorables en las reviews, y en esta etapa priorizamos exclusivamente las señales positivas para construir el ranking.

2. Segundo, normalizamos cada categoría dividiéndola por su valor máximo dentro del conjunto recibido. Esto evita que una categoría con valores absolutos más grandes (por ejemplo, ubicacion, que suele mencionarse mucho) domine el ranking solo por una cuestión de escala: tras la normalización, todas las categorías quedan acotadas al intervalo [0, 1]. 

3. Tercero, calculamos el score final del hotel como la suma de sus valores normalizados en las categorías consideradas.

El resultado es un DataFrame ordenado de mayor a menor `hotel_score`. Este ranking no representa todavía la recomendación final del sistema, sino el ranking estructurado que luego funcionará como Lista B dentro del enfoque híbrido.

In [9]:
def score_hotel(resultado):
    """
    Rankea un DataFrame de hoteles candidatos por la suma de sus scores normalizados
    en las categorías mencionadas. Devuelve el mismo DataFrame ordenado de mayor a
    menor hotel_score.
       - resultado: DataFrame con columnas 'hotel_id_review', 'destino' y una o más columnas de categorías (wifi, vista, limpieza, etc.).
    """
    # Identificamos las columnas de categorías (todo lo que no sea identificador)
    cols_excluir = ["hotel_id_review", "destino"]
    score_cols = [c for c in resultado.columns if c not in cols_excluir]

    # Copia para no modificar el original
    df_scores = resultado.copy()

    # PASO 1: descartamos sentimientos negativos llevándolos a 0
    df_scores[score_cols] = df_scores[score_cols].clip(lower = 0)

    # PASO 2: normalizamos cada categoría a [0, 1] dividiendo por su máximo
    # Así ninguna categoría domina el ranking solo por tener valores más grandes
    for col in score_cols:
        max_val = df_scores[col].max()
        if max_val > 0:
            df_scores[col] = df_scores[col] / max_val
        else:
            df_scores[col] = 0 # Evitamos dividir por cero

    # PASO 3: el score final del hotel es la suma de sus categorías normalizadas
    df_scores["hotel_score"] = df_scores[score_cols].sum(axis = 1)

    # Ordenamos de mayor a menor
    df_scores = df_scores.sort_values("hotel_score", ascending = False)

    return df_scores

Ahora bien, no alcanza con tener una función que rankee. Antes de poder rankear, necesitamos producir los candidatos a rankear. Es decir, un subconjunto de hoteles que cumplen las condiciones del usuario (destino correcto, atributos requeridos con score positivo). La función score_hotel es la pieza que ordena ese subconjunto, pero no sabe construirlo.

La pieza que falta es entonces el mecanismo que traduce la consulta del usuario en lenguaje natural (por ejemplo, "un hotel en Río con buena vista y desayuno, fundamental la ubicación") en una consulta SQL ejecutable sobre la base hoteles.db (en este caso: SELECT ... WHERE destino = 'Brasil - Rio De Janeiro' AND ubicacion > 0 AND (vista > 0 OR desayuno > 0)). Esta traducción es la que va a determinar qué hoteles entran al ranking y bajo qué condiciones.

Para resolver esta traducción replicamos la lógica que ya armamos en el módulo 2, pero adaptada al chatbot. En primer lugar, construimos el prompt QUERY_GENERATOR_PROMPT que le enseña al LLM cómo es la tabla hoteles, cómo armar SQL válido, y cómo distinguir entre atributos que el usuario considera fundamentales (combinables con AND) y atributos que considera deseables (combinables con OR). Este prompt es exactamente el mismo que usamos en el módulo 2 con el modelo local Qwen, pero ahora lo ejecutamos con GPT-4o-mini, lo que nos permite obtener resultados de mejor calidad sin tener que mantener un servidor de inferencia local. En segundo lugar, definimos la función generar_query_sql, que envía la consulta del usuario al LLM junto con el prompt y parsea la respuesta JSON para extraer la query lista para ejecutar.

Con estas tres piezas (la generadora de queries, la base SQLite del módulo 2 y la función `score_hotel`) tenemos todo lo necesario para construir el ranking estructurado. Este ranking será la Lista B del sistema híbrido, es decir, una lista de hoteles candidatos basada en atributos concretos mencionados por el usuario.

In [10]:
# Construimos el texto descriptivo de categorías a partir del YAML
categorias_texto = "\n".join(
    [
        f'- {cat["nombre"]}: score de {cat["descripcion"]}'
        for cat in categorias
    ]
)

# Prompt que guía al LLM para generar queries SQL (mismo prompt del módulo 2)
QUERY_GENERATOR_PROMPT = f"""
Sos un generador de queries SQL. 
Tu tarea es entender las necesidades hoteleras del usuario y traducirlas a una query SQL válida.

IMPORTANTE: el sistema te va a indicar el destino EXACTO a usar al inicio del mensaje del usuario
(en la línea "Destino a usar EXACTAMENTE: '...'"). Tenés que usar ese valor TAL CUAL en el WHERE,
sin modificarlo, sin traducirlo, sin agregar ni quitar tildes ni espacios. Es el formato oficial
del destino en la base de datos.

Debés devolver SOLO un JSON válido con esta estructura:
{{
  "query": "..."
}}

No agregues explicaciones, markdown ni texto adicional.

Nombre de la tabla: hoteles

Columnas disponibles:
- hotel_id_review: identificador del hotel
- destino: lugar donde se encuentra el hotel
{categorias_texto}

Template SQL obligatorio:
SELECT hotel_id_review, destino, <<categorias mencionadas por el usuario>>
FROM hoteles
WHERE destino = "<<destino exacto indicado al inicio del mensaje>>"
AND <<condiciones en base a categorias>>

Reglas para definir las condiciones por categoría:
1) Identificá todas las categorías relevantes mencionadas por el usuario.
2) Clasificá las categorías en dos grupos:

   Categorías IMPORTANTES:
   Son aquellas donde el usuario expresa alta prioridad o necesidad fundamental.
   Ejemplos:
   - "fundamentalmente"
   - "es clave"
   - "muy importante"
   - "necesito sí o sí"
   - "indispensable"

   Estas categorías deben agregarse usando:
   categoria > 0
   y deben combinarse entre sí usando AND.

   Categorías NO IMPORTANTES:
   Son preferencias deseables, pero no excluyentes.
   Ejemplos:
   - "me gustaría"
   - "con buena vista"
   - "ojalá tenga"
   - "preferentemente"
   - "también estaría bueno"

   Estas categorías deben agregarse usando:
   categoria > 0
   y deben combinarse entre sí usando OR dentro de un único bloque.

3) Si existen categorías IMPORTANTES y NO IMPORTANTES al mismo tiempo:
   - Todas las IMPORTANTES deben cumplirse obligatoriamente usando AND.
   - Todas las NO IMPORTANTES deben agruparse en un único bloque OR junto con 1=1.
   - Ese bloque OR debe conectarse con las IMPORTANTES usando AND.

   Ejemplo de condición:
   AND limpieza > 0
   AND (1=1 OR vista > 0 OR desayuno > 0)

4) Si solo existen categorías NO IMPORTANTES:
   Combiná todas usando OR dentro de un único bloque y NO pongas condición 1=1.
   Ejemplo:
   AND (vista > 0 OR desayuno > 0)

5) Si solo existen categorías IMPORTANTES:
   Combiná todas usando AND.
   Ejemplo:
   AND limpieza > 0
   AND ubicacion > 0

Reglas importantes:
- Las categorías NO IMPORTANTES nunca deben combinarse con AND entre sí.
- Todas las categorías NO IMPORTANTES deben agruparse dentro de un único bloque OR junto con 1=1.
- No agregues categorías que el usuario no haya mencionado.
- No inventes columnas.
- Usá solamente columnas incluidas en la lista de columnas disponibles.
- Devolvé únicamente el JSON final.

Ejemplo:

Entrada (mensaje del usuario):
Destino a usar EXACTAMENTE: 'Brasil - Rio De Janeiro'

Consulta del usuario: Quiero un hotel con buena vista y buen desayuno. Es fundamental una buena ubicacion.

Respuesta esperada:
{{
  "query": "SELECT hotel_id_review, destino, ubicacion, vista, desayuno FROM hoteles WHERE destino = 'Brasil - Rio De Janeiro' AND ubicacion > 0 AND (1=1 OR vista > 0 OR desayuno > 0)"
}}
"""

La función generar_query_sql materializa la lógica que acabamos de describir. Recibe la consulta del usuario, la envía al LLM junto con el QUERY_GENERATOR_PROMPT definido arriba, y devuelve únicamente la query SQL ya lista para ejecutar contra la base. La respuesta del LLM viene en formato JSON ({"query": "..."}), por lo que la parseamos para quedarnos solo con el string SQL.

In [11]:
def generar_query_sql(destino, consulta_usuario):
    """
    Genera una query SQL completa a partir del destino validado y la consulta del usuario,
    utilizando el LLM con el prompt del módulo 2.

       - destino: destino validado (ej: 'Brasil - Rio De Janeiro')
       - consulta_usuario: texto libre del usuario (ej: "con buen wifi y desayuno...")
    """
    # Construimos el mensaje del usuario que va al LLM, indicando explícitamente
    # el destino exacto a usar (validado previamente por el agente)
    mensaje = f"Destino a usar EXACTAMENTE: '{destino}' \n Consulta del usuario: {consulta_usuario}"

    # Le pasamos al LLM el prompt del sistema + el mensaje con destino y consulta
    respuesta = llm.invoke([
        {"role": "system", "content": QUERY_GENERATOR_PROMPT},
        {"role": "user", "content": mensaje}
    ])

    # Parseamos el JSON de la respuesta y extraemos la query
    contenido = respuesta.content.strip()
    data = json.loads(contenido)

    return data["query"]

Antes de definir la función recomendar_hoteles_por_atributos, cargamos un mapeo auxiliar entre hotel_id_review y el nombre del hotel. Este mapeo lo extraemos del dataset final del módulo 1 (donde está la metadata original de cada hotel) y nos sirve para enriquecer las recomendaciones con nombres legibles. La base SQLite del módulo 2 solo guarda IDs y scores por categoría, sin nombres, por lo que necesitamos esta tabla auxiliar para presentar resultados que el agente pueda comunicar al usuario.

In [12]:
# Cargamos un mapeo hotel_id → nombre desde el dataset final del módulo 1
# La base SQLite del módulo 2 solo guarda IDs, no nombres, así que lo necesitamos para enriquecer
ruta_dataset_final = Path.cwd().parent / "Data" / "Final" / "eda_final_dataset.parquet"
df_hoteles_nombres = pd.read_parquet(ruta_dataset_final, columns = ["hotel_id_review", "name"])

nombres_por_id = (
    df_hoteles_nombres
    .drop_duplicates("hotel_id_review")
    .set_index("hotel_id_review")["name"]
)

print(f"Mapeo de nombres cargado: {len(nombres_por_id):,} hoteles")

Mapeo de nombres cargado: 2,523 hoteles


Finalmente, encapsulamos el flujo completo del motor estructurado en una única función. Esta función mantiene la lógica original del módulo: genera una query SQL con el LLM, ejecuta la consulta contra SQLite y rankea los hoteles candidatos con `score_hotel`.

La diferencia está en el formato de salida. En lugar de devolver una respuesta final para el agente, ahora devuelve una tabla con el ranking estructurado. Esta tabla representa la Lista B del sistema híbrido y contiene el `structured_score`, su versión normalizada y una marca que indica que el hotel aparece en el motor estructurado.

Luego, esta Lista B será cruzada con la Lista A semántica para construir las recomendaciones principales y las alternativas.

In [13]:
def obtener_ranking_estructurado(destino, consulta_usuario, top_k = 20):
    """
    Genera la Lista B del sistema híbrido: hoteles candidatos rankeados
    por atributos estructurados.

       - destino: destino validado (ej: 'Brasil - Rio De Janeiro')
       - consulta_usuario: texto libre del usuario con atributos concretos
       - top_k: cantidad de hoteles candidatos a conservar
    """
    # Paso 1: el LLM genera la query SQL usando el destino validado
    query = generar_query_sql(destino, consulta_usuario)

    # Paso 2: ejecutamos la query contra la base SQLite
    df_candidatos = pd.read_sql_query(query, conn_hoteles)

    # Si no hay candidatos que cumplan las condiciones, devolvemos DataFrame vacío
    if len(df_candidatos) == 0:
        return pd.DataFrame(columns = [
            "hotel_id_review",
            "hotel_name",
            "destino",
            "structured_score",
            "structured_score_norm",
            "query_sql",
            "aparece_en_estructurado"
        ])

    # Paso 3: aplicamos score_hotel para rankear los candidatos
    df_ranked = score_hotel(df_candidatos).head(top_k).copy()

    # Renombramos hotel_score como structured_score para que sea consistente
    # con el sistema híbrido
    df_ranked["structured_score"] = df_ranked["hotel_score"]

    # Normalizamos el score estructurado dentro de la Lista B
    max_score = df_ranked["structured_score"].max()

    if max_score > 0:
        df_ranked["structured_score_norm"] = df_ranked["structured_score"] / max_score
    else:
        df_ranked["structured_score_norm"] = 0

    # Agregamos nombre legible del hotel
    df_ranked["hotel_name"] = df_ranked["hotel_id_review"].map(nombres_por_id)
    df_ranked["hotel_name"] = df_ranked["hotel_name"].fillna("Nombre no disponible")

    # Guardamos la query para trazabilidad
    df_ranked["query_sql"] = query

    # Marcamos presencia en la lista estructurada
    df_ranked["aparece_en_estructurado"] = True

    return df_ranked.reset_index(drop = True)

In [14]:
# Probamos el motor estructurado
ranking_estructurado_test = obtener_ranking_estructurado(
    destino = "Brasil - Rio De Janeiro",
    consulta_usuario = "con buen desayuno y linda vista",
    top_k = 20
)

# Head
ranking_estructurado_test.head()

,hotel_id_review,destino,desayuno,vista,hotel_score,structured_score,structured_score_norm,hotel_name,query_sql,aparece_en_estructurado
0,231158,Brasil - Rio De Janeiro,1.000000,0.144628,1.144628,1.144628,1.000000,Royal Rio Palace Hotel,"SELECT hotel_id_review, destino, desayuno, vis...",True
1,231814,Brasil - Rio De Janeiro,0.008043,1.000000,1.008043,1.008043,0.880673,Othon Palace Copacabana Rio,"SELECT hotel_id_review, destino, desayuno, vis...",True
2,819124,Brasil - Rio De Janeiro,0.378016,0.619835,0.997851,0.997851,0.871769,Prodigy Santos Dumont by Wish,"SELECT hotel_id_review, destino, desayuno, vis...",True
3,983079,Brasil - Rio De Janeiro,0.431635,0.280992,0.712627,0.712627,0.622584,Laghetto Stilo Barra,"SELECT hotel_id_review, destino, desayuno, vis...",True
4,833230,Brasil - Rio De Janeiro,0.536193,0.173554,0.709747,0.709747,0.620068,Américas Copacabana Hotel,"SELECT hotel_id_review, destino, desayuno, vis...",True


### Motor Semántico (Módulo 3)

En esta sección reutilizamos el motor semántico construido en el módulo 3. Mantenemos el mismo modelo de embeddings y la misma colección de ChromaDB, pero adaptamos la salida de la función.

En lugar de devolver directamente una respuesta final para el agente, esta función devuelve una lista semántica de hoteles candidatos con sus scores. Esta lista será luego cruzada con la lista estructurada para construir el ranking híbrido final.

In [15]:
# Modelo de embeddings (mismo que usamos para indexar las reviews)
modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Conexión a la base Chroma persistida en disco
# El notebook está en Scripts/, por eso usamos Path.cwd().parent para subir a la raíz del proyecto
CHROMA_PATH = Path.cwd().parent / "Data" / "VectorDB"

cliente = chromadb.PersistentClient(path = str(CHROMA_PATH)) # Pasamos la ruta como string porque Chroma espera un path en formato texto
coleccion = cliente.get_collection(name = 'reviews_hoteles')

print(f'Modelo cargado: paraphrase-multilingual-MiniLM-L12-v2 (384 dims)')
print(f'Colección conectada: {coleccion.name} ({coleccion.count():,} vectores)')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo cargado: paraphrase-multilingual-MiniLM-L12-v2 (384 dims)
Colección conectada: reviews_hoteles (251,332 vectores)


Probamos una query real.

In [16]:
# Definimos consulta random
consulta = "hotel tranquilo con linda vista al mar"

# Codificamos la consulta
q_emb = modelo.encode(
    [consulta],
    convert_to_numpy = True,
    normalize_embeddings = True
).astype("float32").tolist()

# Buscamos vectores por similitud
resultado = coleccion.query(
    query_embeddings = q_emb,
    n_results = 5,
    where = {"destino": "Brasil - Rio De Janeiro"},
    include = ["documents", "metadatas", "distances"]
)

resultado

{'ids': [['200466', '185370', '129352', '157224', '139064']],
 'embeddings': None,
 'documents': [['Muy lindo hotel frente al mar, en una playa tranquila. Y sino la pileta con agua calentita',
   'Hermoso hotel frente al mar.',
   'Hotel frente a la playa con excelentes comodidades',
   'El Hotel está muy bueno con vista al mar',
   'La zona donde se encuentra el hotel, segura, tranquila y cerca de la playa']],
 'uris': None,
 'included': ['documents', 'metadatas', 'distances'],
 'data': None,
 'metadatas': [[{'hotel_id': 895491,
    'city': 'Rio De Janeiro',
    'destino': 'Brasil - Rio De Janeiro',
    'country': 'Brasil',
    'hotel_name': 'Hotel Nacional Rio de Janeiro OFICIAL'},
   {'destino': 'Brasil - Rio De Janeiro',
    'hotel_id': 661634,
    'hotel_name': 'Wyndham Rio de Janeiro Barra',
    'city': 'Rio De Janeiro',
    'country': 'Brasil'},
   {'hotel_id': 231670,
    'city': 'Rio De Janeiro',
    'country': 'Brasil',
    'hotel_name': 'Windsor Barra',
    'destino': 'Brasi

Ahora bien, una vez cargado el modelo y hecha la conexión a ChromaDB, definimos la función que genera el ranking semántico. La idea es encapsular la lógica validada en el módulo 3, pero adaptando su salida para que pueda ser usada dentro del sistema híbrido.

La función recibe un destino, una consulta semántica en lenguaje natural y la cantidad de hoteles candidatos a conservar. Internamente vectoriza la consulta, busca reviews similares en ChromaDB filtrando por destino, agrupa los resultados por hotel y aplica la fórmula:

score_semántico = score_promedio × log(1 + reviews_afines)

Este score permite combinar dos señales: qué tan similares son las reviews recuperadas respecto de la consulta y cuánta evidencia existe para cada hotel.

A diferencia del módulo 3, esta función no está pensada para imprimir recomendaciones finales ni para devolver una respuesta directamente al usuario. En este módulo, su objetivo es construir la lista semántica de candidatos, es decir, la Lista A del sistema híbrido. Por eso devuelve una tabla con los hoteles rankeados, sus scores semánticos, la cantidad de reviews afines y algunos ejemplos de reviews relevantes.

Luego, esta lista será normalizada y cruzada con la lista estructurada generada a partir de los atributos del módulo 2. La recomendación final no se decide todavía en esta función, sino en la capa posterior de ranking híbrido.

In [17]:
def obtener_ranking_semantico(consulta, destino, top_k = 20, k_reviews = 300):
    """
    Genera la lista A del sistema híbrido: hoteles candidatos rankeados
    por similitud semántica con la consulta del usuario.

       - consulta: texto libre del usuario o fragmento semántico de la consulta.
       - destino: destino validado, con formato 'País - Ciudad'.
       - top_k: cantidad de hoteles candidatos a conservar.
       - k_reviews: cantidad de reviews recuperadas desde Chroma antes de agrupar por hotel.
    """

    # Vectorizamos la consulta con el mismo modelo usado para indexar las reviews
    q_emb = modelo.encode(
        [consulta],
        convert_to_numpy = True,
        normalize_embeddings = True
    ).astype("float32").tolist()

    # Consultamos Chroma con filtro por destino
    resultado = coleccion.query(
        query_embeddings = q_emb,
        n_results = k_reviews,
        where = {"destino": destino},
        include = ["documents", "metadatas", "distances"]
    )

    # Extraemos resultados
    distancias = resultado["distances"][0]
    documentos = resultado["documents"][0]
    metadatas  = resultado["metadatas"][0]

    # Convertimos distancia coseno a similitud
    similitudes = [1 - d for d in distancias]

    # Armamos DataFrame de reviews recuperadas
    recuperadas = pd.DataFrame({
        "hotel_id_review": [m["hotel_id"] for m in metadatas],
        "hotel_name": [m["hotel_name"] for m in metadatas],
        "texto": documentos,
        "score_review": similitudes
    })

    # Agregamos por hotel
    ranking = (
        recuperadas
        .groupby(["hotel_id_review", "hotel_name"])
        .agg(
            score_promedio = ("score_review", "mean"),
            reviews_afines = ("score_review", "size")
        )
        .reset_index()
    )

    # Score semántico: similitud promedio ponderada por evidencia disponible
    ranking["semantic_score"] = (
        ranking["score_promedio"] * np.log1p(ranking["reviews_afines"])
    )

    # Agregamos ejemplos de reviews para explicar la recomendación
    ejemplos = (
        recuperadas
        .sort_values("score_review", ascending = False)
        .groupby("hotel_id_review")
        .head(2)
        .groupby("hotel_id_review")["texto"]
        .apply(list)
        .reset_index(name = "ejemplos_reviews")
    )

    ranking = ranking.merge(
        ejemplos,
        on = "hotel_id_review",
        how = "left"
    )

    # Ordenamos y nos quedamos con top_k
    ranking = (
        ranking
        .sort_values("semantic_score", ascending = False)
        .head(top_k)
        .reset_index(drop = True)
    )

    # Normalizamos por máximo dentro de la lista semántica
    max_score = ranking["semantic_score"].max()

    if max_score > 0:
        ranking["semantic_score_norm"] = ranking["semantic_score"] / max_score
    else:
        ranking["semantic_score_norm"] = 0

    ranking["aparece_en_semantico"] = True

    return ranking

In [18]:
# Probamos el motor semántico
ranking_semantico_test = obtener_ranking_semantico(
    consulta = "hotel tranquilo para descansar con linda vista al mar",
    destino = "Brasil - Rio De Janeiro",
    top_k = 10,
    k_reviews = 300
)

# Head
ranking_semantico_test.head()

,hotel_id_review,hotel_name,score_promedio,reviews_afines,semantic_score,ejemplos_reviews,semantic_score_norm,aparece_en_semantico
0,661634,Wyndham Rio de Janeiro Barra,0.776011,25,2.528318,"[Hermoso hotel frente al mar., El Hotel está m...",1.000000,True
1,835150,Windsor Oceanico,0.772765,23,2.455890,"[La zona donde se encuentra el hotel, segura, ...",0.971353,True
2,876594,CDesign Hotel,0.775438,18,2.283231,"[Hotel frente al mar, Excelente hotel para des...",0.903063,True
3,895491,Hotel Nacional Rio de Janeiro OFICIAL,0.771455,17,2.229791,"[Muy lindo hotel frente al mar, en una playa t...",0.881927,True
4,231814,Othon Palace Copacabana Rio,0.770183,17,2.226115,"[La ubicación del hotel es súper cómoda, frent...",0.880473,True


### Motor Híbrido

Una vez generados ambos rankings, construimos la capa de integración híbrida.

El primer ranking proviene del motor semántico y ordena hoteles según la similitud entre la consulta del usuario y las reviews vectorizadas. El segundo ranking proviene del motor estructurado y ordena hoteles según atributos concretos extraídos previamente de las reviews.

Aunque conceptualmente podemos pensarlos como dos listas rankeadas de hoteles, en la implementación ambos resultados se manejan como DataFrames, lo que permite cruzarlos por hotel_id_review, normalizar scores, calcular intersecciones y ordenar los resultados finales.
El objetivo de esta etapa es combinar ambas fuentes de evidencia. Primero identificamos los hoteles que aparecen en ambos rankings, ya que cuentan con doble respaldo: son relevantes desde el punto de vista semántico y además tienen buen desempeño en los atributos estructurados consultados. Para esos hoteles calculamos un score híbrido ponderando el score semántico normalizado y el score estructurado normalizado en partes iguales.

Además, conservamos hoteles destacados que aparecen solo en uno de los dos rankings. Estos no forman parte de las recomendaciones principales, pero pueden mostrarse como opciones alternativas: algunos son fuertes por similitud semántica y otros por atributos concretos.

De esta forma, el sistema no depende de que el agente combine resultados manualmente en la redacción, sino que genera una estructura explícita de recomendación híbrida antes de pasarle los resultados al agente.

In [19]:
def obtener_ranking_hibrido(
    destino,
    consulta_semantica,
    consulta_atributos,
    top_k_semantico = 20,
    top_k_estructurado = 20,
    top_n_principales = 5,
    top_n_alternativas = 2,
    k_reviews = 300
):
    """
    Combina el ranking semántico (Lista A) y el ranking estructurado (Lista B)
    para construir una recomendación híbrida.

    La lógica sigue tres pasos:
      1. Genera Lista A con el motor semántico.
      2. Genera Lista B con el motor estructurado.
      3. Cruza ambas listas:
         - A ∩ B: recomendaciones principales.
         - A - B: alternativas semánticas.
         - B - A: alternativas estructuradas.
    """

    # DF A: ranking semántico
    ranking_semantico = obtener_ranking_semantico(
        consulta = consulta_semantica,
        destino = destino,
        top_k = top_k_semantico,
        k_reviews = k_reviews
    )

    # DF B: ranking estructurado
    ranking_estructurado = obtener_ranking_estructurado(
        destino = destino,
        consulta_usuario = consulta_atributos,
        top_k = top_k_estructurado
    )

    # Si alguna lista viene vacía, evitamos errores en el cruce
    if ranking_semantico.empty:
        ranking_semantico = pd.DataFrame(columns = [
            "hotel_id_review",
            "hotel_name",
            "semantic_score",
            "semantic_score_norm",
            "score_promedio",
            "reviews_afines",
            "ejemplos_reviews",
            "aparece_en_semantico"
        ])

    if ranking_estructurado.empty:
        ranking_estructurado = pd.DataFrame(columns = [
            "hotel_id_review",
            "hotel_name",
            "structured_score",
            "structured_score_norm",
            "query_sql",
            "aparece_en_estructurado"
        ])

    # Identificamos hoteles presentes en cada lista
    ids_semantico = set(ranking_semantico["hotel_id_review"])
    ids_estructurado = set(ranking_estructurado["hotel_id_review"])

    ids_ambos = ids_semantico.intersection(ids_estructurado)
    ids_solo_semantico = ids_semantico.difference(ids_estructurado)
    ids_solo_estructurado = ids_estructurado.difference(ids_semantico)

    # Recomendaciones principales: hoteles presentes en ambas listas
    recomendaciones_principales = (
        ranking_semantico
        .merge(
            ranking_estructurado,
            on = "hotel_id_review",
            how = "inner",
            suffixes = ("_semantico", "_estructurado")
        )
    )

    if not recomendaciones_principales.empty:
        # Unificamos nombre del hotel
        recomendaciones_principales["hotel_name"] = recomendaciones_principales[
            "hotel_name_semantico"
        ].fillna(
            recomendaciones_principales["hotel_name_estructurado"]
        )

        # Score híbrido según la propuesta: promedio ponderado
        recomendaciones_principales["hybrid_score"] = (
            0.5 * recomendaciones_principales["semantic_score_norm"]
            + 0.5 * recomendaciones_principales["structured_score_norm"]
        ) # Promedio ponderado 50/50, esto se puede ajustar según se le quiera dar más importancia a una que a otra

        recomendaciones_principales = (
            recomendaciones_principales
            .sort_values("hybrid_score", ascending = False)
            .head(top_n_principales)
            .reset_index(drop = True)
        )

    # Alternativas semánticas: aparecen solo en Lista A
    alternativas_semanticas = (
        ranking_semantico[
            ranking_semantico["hotel_id_review"].isin(ids_solo_semantico)
        ]
        .sort_values("semantic_score_norm", ascending = False)
        .head(top_n_alternativas)
        .reset_index(drop = True)
    )

    # Alternativas estructuradas: aparecen solo en Lista B
    alternativas_estructuradas = (
        ranking_estructurado[
            ranking_estructurado["hotel_id_review"].isin(ids_solo_estructurado)
        ]
        .sort_values("structured_score_norm", ascending = False)
        .head(top_n_alternativas)
        .reset_index(drop = True)
    )

    return {
        "destino": destino,
        "consulta_semantica": consulta_semantica,
        "consulta_atributos": consulta_atributos,
        "ranking_semantico": ranking_semantico,
        "ranking_estructurado": ranking_estructurado,
        "recomendaciones_principales": recomendaciones_principales,
        "alternativas_semanticas": alternativas_semanticas,
        "alternativas_estructuradas": alternativas_estructuradas
    }

In [20]:
# Probamos la función híbrida
resultado_hibrido_test = obtener_ranking_hibrido(
    destino = "Brasil - Rio De Janeiro",
    consulta_semantica = "hotel tranquilo para descansar con linda vista al mar",
    consulta_atributos = "con buen desayuno y linda vista",
    top_k_semantico = 20,
    top_k_estructurado = 20,
    top_n_principales = 5,
    top_n_alternativas = 2,
    k_reviews = 300
)

In [21]:
# Imprimimos recomendaciones principales
resultado_hibrido_test["recomendaciones_principales"]

,hotel_id_review,hotel_name_semantico,score_promedio,reviews_afines,semantic_score,ejemplos_reviews,semantic_score_norm,aparece_en_semantico,destino,vista,desayuno,hotel_score,structured_score,structured_score_norm,hotel_name_estructurado,query_sql,aparece_en_estructurado,hotel_name,hybrid_score
0,231814,Othon Palace Copacabana Rio,0.770183,17,2.226115,"[La ubicación del hotel es súper cómoda, frent...",0.880473,True,Brasil - Rio De Janeiro,1.000000,0.008043,1.008043,1.008043,0.880673,Othon Palace Copacabana Rio,"SELECT hotel_id_review, destino, vista, desayu...",True,Othon Palace Copacabana Rio,0.880573
1,231158,Royal Rio Palace Hotel,0.767378,10,1.840091,"[Hotel muy bien ubicado, muy céntrico y a cuad...",0.727793,True,Brasil - Rio De Janeiro,0.144628,1.000000,1.144628,1.144628,1.000000,Royal Rio Palace Hotel,"SELECT hotel_id_review, destino, vista, desayu...",True,Royal Rio Palace Hotel,0.863896
2,835150,Windsor Oceanico,0.772765,23,2.455890,"[La zona donde se encuentra el hotel, segura, ...",0.971353,True,Brasil - Rio De Janeiro,0.309917,0.302949,0.612866,0.612866,0.535428,Windsor Oceanico,"SELECT hotel_id_review, destino, vista, desayu...",True,Windsor Oceanico,0.753391
3,661634,Wyndham Rio de Janeiro Barra,0.776011,25,2.528318,"[Hermoso hotel frente al mar., El Hotel está m...",1.000000,True,Brasil - Rio De Janeiro,0.557851,0.000000,0.557851,0.557851,0.487365,Wyndham Rio de Janeiro Barra,"SELECT hotel_id_review, destino, vista, desayu...",True,Wyndham Rio de Janeiro Barra,0.743682
4,819124,Prodigy Santos Dumont by Wish,0.756376,6,1.471839,[Buenos los desayunos y la tranquilidad en el ...,0.582142,True,Brasil - Rio De Janeiro,0.619835,0.378016,0.997851,0.997851,0.871769,Prodigy Santos Dumont by Wish,"SELECT hotel_id_review, destino, vista, desayu...",True,Prodigy Santos Dumont by Wish,0.726955


### Definición De Las Tools Del Agente

Con los dos motores listos y la función de ranking híbrido definida, construimos las herramientas (`tools`) que el agente puede invocar. Cada tool es una función de Python decorada con `@tool` de LangChain, lo que permite que el LLM la vea y la llame cuando necesita recuperar información del sistema.

Para que el LLM use correctamente las tools, hay dos elementos importantes:

1. El nombre de la función debe ser descriptivo, porque el LLM lo utiliza para entender cuándo corresponde llamarla.

2. El docstring debe explicar claramente qué hace la tool, qué espera recibir y qué devuelve. En la práctica, el LLM se apoya fuertemente en esta descripción para decidir cómo usarla.

En esta versión, la arquitectura expone dos tools principales:

- `listar_destinos_disponibles`: devuelve la lista completa de destinos disponibles en el sistema. El agente la utiliza cuando necesita validar si el destino mencionado por el usuario existe en la base o cuando debe sugerir alternativas cercanas.

- `buscar_hoteles_hibrido`: encapsula la lógica principal del recomendador. Recibe el destino, una consulta semántica en lenguaje natural y una consulta orientada a atributos. Internamente genera la Lista A con el motor semántico, genera la Lista B con el motor estructurado, cruza ambas listas y devuelve recomendaciones principales junto con alternativas.

La combinación entre el motor semántico y el motor estructurado ocurre dentro de la función híbrida, antes de que el LLM redacte la respuesta final.

De esta forma, el rol del agente queda más acotado y controlado: interpreta la consulta del usuario, valida el destino, llama a la herramienta híbrida con los parámetros adecuados y presenta los resultados de forma conversacional.

Antes de definir la primera tool, necesitamos tener cargada la lista de destinos. Es un insumo que la tool va a devolver, pero conviene cargarlo una sola vez al inicio del notebook y no cada vez que el agente la consulta.

In [22]:
# Importamos librería
from langchain_core.tools import tool

# Recuperamos la lista de destinos válidos desde el dataset final del módulo 1
ruta_dataset_final = Path.cwd().parent / "Data" / "Final" / "eda_final_dataset.parquet" # Es una operación rápida y la hacemos una sola vez al cargar el notebook
df = pd.read_parquet(ruta_dataset_final)

destinos_disponibles = sorted(df["destino"].unique().tolist())

print(f"Destinos disponibles: {len(destinos_disponibles)}")
print(destinos_disponibles[:5], "...")

Destinos disponibles: 67
['Argentina - Buenos Aires', 'Argentina - Córdoba', 'Argentina - El Calafate', 'Argentina - Mar Del Plata', 'Argentina - Mendoza'] ...


La idea es definir la lista de destinos afuera de la tool por varias razones:

1. Eficiencia. La lista de destinos no cambia entre consultas. Si la metieramos adentro de la función, cada vez que el LLM llamara a la tool, se ejecutaría el pd.read_parquet(...) desde cero e implica leer un archivo del disco cada vez que el usuario pregunta. Cargándola una sola vez al inicio del notebook, la función simplemente devuelve la variable ya cargada (instantáneo).

2. Claridad. Separa "preparar los datos" de "exponerlos como herramienta".

3. Reutilización. La variable destinos_disponibles la podemos usar también en otros lugares del notebook (validar inputs, mostrarla en logs, etc.), no solo dentro de la tool.

Ahora si, definimos las tools del agente.

In [23]:
@tool
def listar_destinos_disponibles() -> list:
    """
    Devuelve la lista completa de destinos turísticos disponibles en el sistema.
    Cada destino tiene el formato 'País - Ciudad' (ej: 'Brasil - Rio De Janeiro').

    Usar esta tool cuando el usuario menciona un destino y necesitás verificar
    si está disponible, o cuando el usuario pide saber qué destinos hay.
    """

    return destinos_disponibles

@tool
def buscar_hoteles_hibrido(
    destino: str,
    consulta_semantica: str,
    consulta_atributos: str,
    top_n_principales: int = 5,
    top_n_alternativas: int = 2
) -> dict:
    """
    Recomienda hoteles usando el sistema híbrido.

    Esta tool debe usarse cuando el usuario pida recomendaciones de hoteles
    dentro de un destino.

    Parámetros:
    - destino: destino validado con formato 'País - Ciudad'.
    - consulta_semantica: parte descriptiva o subjetiva de la consulta del usuario.
      Ejemplos: 'hotel tranquilo para descansar', 'linda vista al mar',
      'ideal para una pareja', 'ambiente familiar'.
    - consulta_atributos: parte de la consulta relacionada con atributos concretos.
      Ejemplos: 'con buen desayuno y vista', 'con wifi y buena ubicación',
      'fundamental limpieza y seguridad'.
    - top_n_principales: cantidad de hoteles principales a devolver.
    - top_n_alternativas: cantidad de alternativas semánticas y estructuradas a devolver.

    Internamente:
    1. Genera la Lista A con el motor semántico.
    2. Genera la Lista B con el motor estructurado.
    3. Cruza ambas listas por hotel_id_review.
    4. Prioriza hoteles presentes en ambas listas usando hybrid_score.
    5. Devuelve alternativas que aparecen solo en una de las dos listas.

    Devuelve un diccionario con:
    - recomendaciones_principales
    - alternativas_semanticas
    - alternativas_estructuradas
    """

    resultado = obtener_ranking_hibrido(
        destino = destino,
        consulta_semantica = consulta_semantica,
        consulta_atributos = consulta_atributos,
        top_n_principales = top_n_principales,
        top_n_alternativas = top_n_alternativas
    )

    return {
        "destino": resultado["destino"],
        "consulta_semantica": resultado["consulta_semantica"],
        "consulta_atributos": resultado["consulta_atributos"],
        "recomendaciones_principales": resultado["recomendaciones_principales"].to_dict(orient = "records"),
        "alternativas_semanticas": resultado["alternativas_semanticas"].to_dict(orient = "records"),
        "alternativas_estructuradas": resultado["alternativas_estructuradas"].to_dict(orient = "records")
    }


# Agrupamos las tools en una lista para pasársela al agente
tools = [
    listar_destinos_disponibles,
    buscar_hoteles_hibrido
]

print(f"Tools registradas: {[t.name for t in tools]}")

Tools registradas: ['listar_destinos_disponibles', 'buscar_hoteles_hibrido']


### Armado Del Agente Con LangChain

Con las tools definidas, armamos el agente que va a orquestar la interacción con el usuario. El agente combina tres elementos:

1. El **LLM** (`GPT-4o-mini`), que interpreta la consulta del usuario, identifica el destino, separa la intención semántica de los atributos concretos y redacta la respuesta final.

2. Las **tools** disponibles, que el LLM puede invocar. En esta versión exponemos una tool para validar destinos y una tool principal de recomendación híbrida.

3. Un **prompt del sistema**, que le indica al LLM cómo debe comportarse, cómo validar destinos, cómo construir los argumentos de la tool híbrida y cómo presentar los resultados.

LangChain provee la función `create_tool_calling_agent`, que ensambla el LLM, las tools y el prompt. Luego, `AgentExecutor` se encarga del loop de razonamiento: el LLM decide qué tool llamar, LangChain la ejecuta, devuelve el resultado al LLM, y el ciclo continúa hasta que el agente tiene suficiente información para responder.

La diferencia principal respecto de la versión anterior es que el agente ya no decide entre un motor semántico y un motor estructurado como herramientas separadas. Ahora llama a una tool híbrida que internamente genera la Lista A semántica, la Lista B estructurada, cruza ambos rankings y devuelve recomendaciones principales más alternativas.

Activamos `verbose = True` durante el desarrollo para visualizar qué tools decide llamar el agente, con qué argumentos y qué resultados recibe. Esto ayuda a depurar el comportamiento y a defender cómo se construye la recomendación final.

In [24]:
# Importamos librerías
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

# Prompt del sistema: define el rol y comportamiento del agente
system_prompt = """Sos un asistente experto en recomendación de hoteles para turistas que viajan por Latinoamérica.

Tu trabajo es interpretar lo que el usuario busca y usar las herramientas disponibles para darle recomendaciones explicables y personalizadas.

REGLAS DE USO:

1. Siempre que el usuario mencione un destino, validá que existe usando 'listar_destinos_disponibles'.
   - Los destinos tienen el formato 'País - Ciudad' (ej: 'Brasil - Rio De Janeiro').
   - Si el usuario menciona solo la ciudad (ej: 'Río'), inferí el destino completo basándote en la lista.
   - Si el destino no existe en el sistema, decíselo amablemente y sugerí destinos similares disponibles.

2. Para recomendar hoteles, usá la herramienta 'buscar_hoteles_hibrido'.
   Esta es la herramienta principal del sistema de recomendación.

3. Antes de llamar a 'buscar_hoteles_hibrido', separá la consulta del usuario en dos partes:

   - consulta_semantica:
     Debe incluir la intención general, descriptiva o subjetiva del usuario.
     No la reduzcas demasiado. Conservá el contexto descriptivo, subjetivo o experiencial
     de la consulta, incluyendo palabras como tranquilo, romántico, familiar, cómodo,
     cerca de la playa, vista al mar, ideal para trabajar o para descansar.
     
     La consulta semántica puede repetir algunos conceptos que también aparezcan en
     los atributos estructurados si esos conceptos ayudan a capturar mejor la intención
     del usuario en las reviews.
     
     Ejemplos: 'hotel tranquilo para descansar', 'linda vista al mar',
     'romántico para una pareja', 'ambiente familiar', 'cómodo para vacaciones'.

   - consulta_atributos:
     Debe incluir solo los atributos concretos mencionados por el usuario que puedan
     mapearse a categorías estructuradas del sistema, como wifi, vista, ubicación,
     ruido/tranquilidad, limpieza, desayuno, habitación/comodidad, seguridad,
     aire acondicionado, estacionamiento, check-in, restaurante o transporte.
     
     Mantené palabras como 'fundamental', 'imprescindible' o 'importante' si el usuario
     las menciona, porque pueden modificar cómo el motor estructurado genera la  query SQL.

4. No llames por separado al motor semántico ni al motor estructurado.
   La combinación entre ambos ya ocurre dentro de 'buscar_hoteles_hibrido'.

5. La herramienta híbrida devuelve tres grupos:
   - recomendaciones_principales: hoteles que aparecen tanto en el ranking semántico como en el ranking estructurado.
   - alternativas_semanticas: hoteles fuertes por similitud semántica, aunque no aparezcan en el ranking estructurado.
   - alternativas_estructuradas: hoteles fuertes por atributos concretos, aunque no aparezcan en el ranking semántico.

6. Al presentar las recomendaciones:
   - Mostrá primero las recomendaciones principales.
   - Explicá brevemente por qué cada hotel es relevante.
   - Cuando sea posible, mencioná que el hotel combina evidencia semántica y atributos estructurados.
   - Después, si hay alternativas, podés presentarlas como "otras opciones que también podrían interesarte".
   - No muestres scores técnicos salvo que el usuario los pida explícitamente.

7. Si no hay recomendaciones principales suficientes, podés apoyarte en las alternativas semánticas y estructuradas, aclarando que son opciones destacadas por una de las dos señales.

8. Si el usuario te pide cosas que no podés hacer, como reservar, mostrar precios, fotos o disponibilidad en tiempo real, aclará amablemente que sos un asistente de recomendación basado en reviews y que esa información no está disponible en el sistema.

9. Respondé siempre en español, salvo que el usuario te hable en portugués.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# Armamos el agente con las tools y el prompt
agente = create_tool_calling_agent(llm, tools, prompt)

# El executor es quien efectivamente corre el loop de razonamiento
executor = AgentExecutor(
    agent = agente,
    tools = tools,
    verbose = True, # Mostramos paso a paso lo que decide el agente
    max_iterations = 5 # Cota de seguridad para evitar loops infinitos
)

print("Agente listo.")

Agente listo.


### Prueba Del Agente

Probamos el agente con una consulta realista. Con verbose = True vamos a ver en consola exactamente qué decide hacer en cada paso: qué tool llama, con qué argumentos, qué le devuelve, y cómo combina la información para producir la respuesta final.

In [25]:
# Prompt del usuario
respuesta = executor.invoke({
    "input": "Quiero un hotel tranquilo en Río de Janeiro con linda vista al mar y rico desayuno."
})

print("\n" + "=" * 70)
print("RESPUESTA FINAL DEL AGENTE:")
print("=" * 70)
print(respuesta["output"])



> Entering new AgentExecutor chain...

Invoking: `listar_destinos_disponibles` with `{}`


['Argentina - Buenos Aires', 'Argentina - Córdoba', 'Argentina - El Calafate', 'Argentina - Mar Del Plata', 'Argentina - Mendoza', 'Argentina - Puerto Iguazú', 'Argentina - Rosario', 'Argentina - Salta', 'Argentina - San Carlos De Bariloche', 'Argentina - Ushuaia', 'Argentina - Villa Carlos Paz', 'Brasil - Aracaju', 'Brasil - Arraial Do Cabo', 'Brasil - Arraial D´ajuda', 'Brasil - Balneário Camboriú', 'Brasil - Belo Horizonte', 'Brasil - Brasilia', 'Brasil - Búzios', 'Brasil - Campos Do Jordão', 'Brasil - Curitiba', 'Brasil - Florianópolis', 'Brasil - Fortaleza', 'Brasil - Foz De Iguazú', 'Brasil - Gramado', 'Brasil - Joao Pessoa', 'Brasil - Maceió', 'Brasil - Maragogi', 'Brasil - Morro De San Pablo', 'Brasil - Natal', 'Brasil - Peña', 'Brasil - Porto Alegre', 'Brasil - Porto De Galinhas', 'Brasil - Porto Seguro', 'Brasil - Praia Da Pipa', 'Brasil - Recife', 'Brasil - Rio De Janeiro', 'Brasil -

Probamos también con una consulta más rica para validar la lógica de atributos importantes y deseables. En este caso el usuario pide 'buen desayuno' (deseable) y 'fundamental wifi' (importante), lo cual debería traducirse en una query SQL con la lógica wifi > 0 AND (1=1 OR desayuno > 0) y privilegiar hoteles que combinen bien ambas señales con buena ubicación semántica.

In [26]:
# Prompt del usuario
respuesta = executor.invoke({
    "input": "Estoy planeando un viaje a Mar Del Plata. Quiero un hotel bien ubicado, con un buen desayuno, y es fundamental que tenga buen wifi ya que necesito trabajar."
})

print("\n" + "=" * 70)
print("RESPUESTA FINAL DEL AGENTE:")
print("=" * 70)
print(respuesta["output"])



> Entering new AgentExecutor chain...

Invoking: `listar_destinos_disponibles` with `{}`


['Argentina - Buenos Aires', 'Argentina - Córdoba', 'Argentina - El Calafate', 'Argentina - Mar Del Plata', 'Argentina - Mendoza', 'Argentina - Puerto Iguazú', 'Argentina - Rosario', 'Argentina - Salta', 'Argentina - San Carlos De Bariloche', 'Argentina - Ushuaia', 'Argentina - Villa Carlos Paz', 'Brasil - Aracaju', 'Brasil - Arraial Do Cabo', 'Brasil - Arraial D´ajuda', 'Brasil - Balneário Camboriú', 'Brasil - Belo Horizonte', 'Brasil - Brasilia', 'Brasil - Búzios', 'Brasil - Campos Do Jordão', 'Brasil - Curitiba', 'Brasil - Florianópolis', 'Brasil - Fortaleza', 'Brasil - Foz De Iguazú', 'Brasil - Gramado', 'Brasil - Joao Pessoa', 'Brasil - Maceió', 'Brasil - Maragogi', 'Brasil - Morro De San Pablo', 'Brasil - Natal', 'Brasil - Peña', 'Brasil - Porto Alegre', 'Brasil - Porto De Galinhas', 'Brasil - Porto Seguro', 'Brasil - Praia Da Pipa', 'Brasil - Recife', 'Brasil - Rio De Janeiro', 'Brasil -

### Evaluación Cualitativa Del Agente Recomendador

Para poder analizar el comportamiento del agente de forma más transparente, definimos una versión del `AgentExecutor` orientada a evaluación. A diferencia del executor usado para interactuar con el usuario final, este executor devuelve los pasos intermedios de ejecución. Esto nos permite inspeccionar qué tool decidió llamar el agente, con qué argumentos lo hizo y cómo llegó a la respuesta final.

Además, configuramos `verbose = False` para evitar imprimir todo el razonamiento en pantalla durante la ejecución masiva de casos, y usamos `max_iterations = 5` para limitar la cantidad máxima de ciclos de decisión que puede realizar el agente antes de detenerse. En este contexto, una iteración corresponde a un ciclo en el que el agente decide si necesita llamar una herramienta, ejecuta esa herramienta y recibe el resultado para continuar razonando.

In [27]:

executor_eval = AgentExecutor(
    agent = agente,
    tools = tools,
    verbose = False,
    max_iterations = 5,
    return_intermediate_steps = True # Esto permite recuperar programáticamente qué tool llamó el agente y con qué argumentos
)

Definimos tres casos de evaluación para analizar cómo se comporta el agente frente a consultas realistas de usuarios. Los prompts combinan destinos disponibles con distintos tipos de preferencias: algunas más concretas, como wifi, limpieza o desayuno, y otras más subjetivas, como tranquilidad, romanticismo o buena vista.

Estos casos nos permiten auditar la lógica del agente y revisar cómo transforma cada consulta en una llamada a la herramienta híbrida de recomendación.

In [28]:
# Definimos casos de evaluación
casos_eval = pd.DataFrame([
    {
        "caso_id": 1,
        "prompt_usuario": "Quiero un hotel tranquilo en Río de Janeiro con una linda vista al mar y rico desayuno."
    },
    {
        "caso_id": 2,
        "prompt_usuario": "Estoy planeando un viaje a Mar Del Plata. Necesito un hotel bien ubicado, con un buen desayuno, y es fundamental que tenga buen wifi ya que necesito trabajar."
    },
    {
        "caso_id": 3,
        "prompt_usuario": "Quiero un hotel en Florianopolis, cerca de la playa, y con aire acondicionado."
    }
])

Para poder analizar cada ejecución del agente, necesitamos extraer de los `intermediate_steps` la llamada concreta que hizo a la herramienta híbrida de recomendación. Por eso definimos una función auxiliar que recorre los pasos intermedios del agente y busca una tool específica por nombre.

La función devuelve dos elementos: la acción del agente, donde podemos ver qué herramienta llamó y con qué argumentos, y la observación, que corresponde al resultado devuelto por esa herramienta. Si el agente no llamó a esa tool en una ejecución determinada, la función devuelve `None`.

In [29]:
def buscar_paso_tool(respuesta_agente, nombre_tool):
    """
    Busca dentro de intermediate_steps la llamada a una tool específica.
    Devuelve la acción del agente y la observación devuelta por la tool.
    """

    for action, observation in respuesta_agente.get("intermediate_steps", []):
        if action.tool == nombre_tool:
            return action, observation

    return None, None

Definimos una función auxiliar para recuperar la query SQL utilizada por el motor estructurado en cada caso de evaluación. Esto nos permite auditar cómo se tradujeron los atributos del usuario (por ejemplo wifi, desayuno, limpieza o ubicación) en una consulta concreta sobre la base de hoteles.

La función busca este campo dentro de las recomendaciones principales y de las alternativas estructuradas, ya que la SQL puede aparecer en cualquiera de esos buckets según el resultado de la tool híbrida.

In [30]:
def extraer_query_sql(observacion):
    """
    Extrae la query SQL usada por el motor estructurado.
    Busca en recomendaciones principales y alternativas estructuradas.
    """

    buckets = [
        "recomendaciones_principales",
        "alternativas_estructuradas"
    ]

    for bucket in buckets:
        resultados = observacion.get(bucket, [])

        for item in resultados:
            if "query_sql" in item:
                return item["query_sql"]

    return None

Creamos una función de evaluación que ejecuta el agente sobre todos los prompts definidos y registra sus decisiones principales. Para cada caso, guardamos si llamó a la tool híbrida, qué destino detectó, qué consulta envió al motor semántico, qué atributos envió al motor estructurado, qué SQL se generó y cuántas recomendaciones devolvió cada bucket.

El objetivo es transformar la ejecución del agente en una tabla de trazabilidad, para poder analizar de forma clara cómo interpreta cada consulta y cómo usa la herramienta de recomendación.

In [31]:
def evaluar_traza_agente(casos_eval):
    """
    Ejecuta el agente sobre varios prompts y guarda qué decisiones tomó:
    destino, consulta semántica, consulta de atributos, SQL generado,
    cantidad de recomendaciones y respuesta final.
    """

    filas = []

    for _, caso in casos_eval.iterrows():

        respuesta = executor_eval.invoke({
            "input": caso["prompt_usuario"]
        })

        action_hibrido, observacion_hibrido = buscar_paso_tool(
            respuesta,
            "buscar_hoteles_hibrido"
        )

        if action_hibrido is None:
            filas.append({
                "caso_id": caso["caso_id"],
                "prompt_usuario": caso["prompt_usuario"],
                "tool_hibrida_llamada": False,
                "respuesta_final": respuesta.get("output")
            })
            continue

        tool_input = action_hibrido.tool_input

        filas.append({
            "caso_id": caso["caso_id"],
            "prompt_usuario": caso["prompt_usuario"],
            "tool_hibrida_llamada": True,
            "destino_enviado": tool_input.get("destino"),
            "consulta_semantica_enviada": tool_input.get("consulta_semantica"),
            "consulta_atributos_enviada": tool_input.get("consulta_atributos"),
            "query_sql": extraer_query_sql(observacion_hibrido),
            "n_recomendaciones_principales": len(
                observacion_hibrido.get("recomendaciones_principales", [])
            ),
            "n_alternativas_semanticas": len(
                observacion_hibrido.get("alternativas_semanticas", [])
            ),
            "n_alternativas_estructuradas": len(
                observacion_hibrido.get("alternativas_estructuradas", [])
            ),
            "respuesta_final": respuesta.get("output")
        })

    return pd.DataFrame(filas)

Definimos una función auxiliar para convertir las recomendaciones devueltas por la tool híbrida en una tabla larga. En lugar de trabajar con una respuesta anidada por buckets, esta función genera una fila por cada hotel recomendado.

Esto nos permite analizar con mayor detalle qué hoteles aparecieron como recomendaciones principales, alternativas semánticas o alternativas estructuradas, junto con sus scores, atributos relevantes, ejemplos de reviews y posición en el ranking.

In [32]:
def flatten_recomendaciones(caso_id, prompt_usuario, observacion_hibrido):
    """
    Convierte las recomendaciones del sistema híbrido en una tabla larga:
    una fila por hotel recomendado.
    """

    filas = []

    buckets = {
        "principal": "recomendaciones_principales",
        "alternativa_semantica": "alternativas_semanticas",
        "alternativa_estructurada": "alternativas_estructuradas"
    }

    for tipo_recomendacion, bucket in buckets.items():

        recomendaciones = observacion_hibrido.get(bucket, [])

        for posicion, item in enumerate(recomendaciones, start = 1):

            filas.append({
                "caso_id": caso_id,
                "prompt_usuario": prompt_usuario,
                "tipo_recomendacion": tipo_recomendacion,
                "posicion": posicion,
                "hotel_id_review": item.get("hotel_id_review"),
                "hotel_name": item.get("hotel_name"),
                "semantic_score": item.get("semantic_score"),
                "semantic_score_norm": item.get("semantic_score_norm"),
                "structured_score": item.get("structured_score"),
                "structured_score_norm": item.get("structured_score_norm"),
                "hybrid_score": item.get("hybrid_score"),
                "score_promedio": item.get("score_promedio"),
                "reviews_afines": item.get("reviews_afines"),
                "ejemplos_reviews": item.get("ejemplos_reviews"),
                "query_sql": item.get("query_sql"),
                "vista": item.get("vista"),
                "desayuno": item.get("desayuno"),
                "ubicacion": item.get("ubicacion"),
                "limpieza": item.get("limpieza"),
                "wifi": item.get("wifi"),
                "ruido": item.get("ruido"),
                "habitacion": item.get("habitacion"),
                "personal": item.get("personal"),
                "seguridad": item.get("seguridad"),
                "aire_acondicionado": item.get("aire_acondicionado"),
                "estacionamiento": item.get("estacionamiento"),
                "checkin": item.get("checkin"),
                "restaurante": item.get("restaurante"),
                "transporte": item.get("transporte")
            })

    return filas

Finalmente, integramos la evaluación del agente y el análisis de recomendaciones en una única función. Esta función ejecuta cada prompt de evaluación, recupera la llamada a la tool híbrida y construye dos tablas complementarias.

* La primera tabla resume la interpretación del agente a nivel prompt: destino detectado, consulta semántica, consulta de atributos, SQL generada y cantidad de resultados por tipo de recomendación. 

* La segunda tabla baja al nivel de hotel recomendado, permitiendo analizar los scores, atributos y posición de cada hotel dentro de cada bucket.

De esta forma, obtenemos una visión completa del comportamiento del sistema: por un lado, cómo razona el agente; por otro, qué recomendaciones concretas produce.

In [33]:
def evaluar_recomendaciones_agente(casos_eval):
    """
    Ejecuta el agente y devuelve dos DataFrames:
       1. df_eval_prompts: decisiones del agente por prompt.
       2. df_eval_hoteles: recomendaciones y scores por hotel.
    """

    filas_prompts = []
    filas_hoteles = []

    for _, caso in casos_eval.iterrows():

        respuesta = executor_eval.invoke({
            "input": caso["prompt_usuario"]
        })

        action_hibrido, observacion_hibrido = buscar_paso_tool(
            respuesta,
            "buscar_hoteles_hibrido"
        )

        if action_hibrido is None:
            filas_prompts.append({
                "caso_id": caso["caso_id"],
                "prompt_usuario": caso["prompt_usuario"],
                "tool_hibrida_llamada": False,
                "respuesta_final": respuesta.get("output")
            })
            continue

        tool_input = action_hibrido.tool_input

        filas_prompts.append({
            "caso_id": caso["caso_id"],
            "prompt_usuario": caso["prompt_usuario"],
            "tool_hibrida_llamada": True,
            "destino_enviado": tool_input.get("destino"),
            "consulta_semantica_enviada": tool_input.get("consulta_semantica"),
            "consulta_atributos_enviada": tool_input.get("consulta_atributos"),
            "query_sql": extraer_query_sql(observacion_hibrido),
            "n_recomendaciones_principales": len(
                observacion_hibrido.get("recomendaciones_principales", [])
            ),
            "n_alternativas_semanticas": len(
                observacion_hibrido.get("alternativas_semanticas", [])
            ),
            "n_alternativas_estructuradas": len(
                observacion_hibrido.get("alternativas_estructuradas", [])
            ),
            "respuesta_final": respuesta.get("output")
        })

        filas_hoteles.extend(
            flatten_recomendaciones(
                caso_id = caso["caso_id"],
                prompt_usuario = caso["prompt_usuario"],
                observacion_hibrido = observacion_hibrido
            )
        )

    df_eval_prompts = pd.DataFrame(filas_prompts)
    df_eval_hoteles = pd.DataFrame(filas_hoteles)

    return df_eval_prompts, df_eval_hoteles

In [34]:
# Corremos las evaluaciones
df_eval_prompts, df_eval_hoteles = evaluar_recomendaciones_agente(casos_eval)

In [35]:
# Mostramos la tabla de razonamiento del agente
df_eval_prompts

,caso_id,prompt_usuario,tool_hibrida_llamada,destino_enviado,consulta_semantica_enviada,consulta_atributos_enviada,query_sql,n_recomendaciones_principales,n_alternativas_semanticas,n_alternativas_estructuradas,respuesta_final
0,1,Quiero un hotel tranquilo en Río de Janeiro co...,True,Brasil - Rio De Janeiro,hotel tranquilo con linda vista al mar y rico ...,desayuno,"SELECT hotel_id_review, destino, desayuno FROM...",5,2,2,Aquí tienes algunas recomendaciones de hoteles...
1,2,Estoy planeando un viaje a Mar Del Plata. Nece...,True,Argentina - Mar Del Plata,"hotel bien ubicado, con buen desayuno, y funda...","bien ubicado, buen desayuno, fundamental wifi","SELECT hotel_id_review, destino, wifi, ubicaci...",1,2,0,He encontrado algunas recomendaciones de hotel...
2,3,"Quiero un hotel en Florianopolis, cerca de la ...",True,Brasil - Florianópolis,hotel cerca de la playa,aire acondicionado,"SELECT hotel_id_review, destino, aire_acondici...",5,2,2,Aquí tienes algunas recomendaciones de hoteles...


In [36]:
# Mostramos la tabla de hoteles recomendados
df_eval_hoteles

,caso_id,prompt_usuario,tipo_recomendacion,posicion,hotel_id_review,hotel_name,semantic_score,semantic_score_norm,structured_score,structured_score_norm,...,wifi,ruido,habitacion,personal,seguridad,aire_acondicionado,estacionamiento,checkin,restaurante,transporte
0,1,Quiero un hotel tranquilo en Río de Janeiro co...,principal,1,231158,Royal Rio Palace Hotel,2.146544,0.821549,1.000000,1.000000,...,NaN,None,None,None,None,NaN,None,None,None,None
1,1,Quiero un hotel tranquilo en Río de Janeiro co...,principal,2,833230,Américas Copacabana Hotel,2.223474,0.850992,0.536193,0.536193,...,NaN,None,None,None,None,NaN,None,None,None,None
2,1,Quiero un hotel tranquilo en Río de Janeiro co...,principal,3,983079,Laghetto Stilo Barra,2.372409,0.907994,0.431635,0.431635,...,NaN,None,None,None,None,NaN,None,None,None,None
3,1,Quiero un hotel tranquilo en Río de Janeiro co...,principal,4,835150,Windsor Oceanico,2.612802,1.000000,0.302949,0.302949,...,NaN,None,None,None,None,NaN,None,None,None,None
4,1,Quiero un hotel tranquilo en Río de Janeiro co...,principal,5,819124,Prodigy Santos Dumont by Wish,1.935083,0.740616,0.378016,0.378016,...,NaN,None,None,None,None,NaN,None,None,None,None
5,1,Quiero un hotel tranquilo en Río de Janeiro co...,alternativa_semantica,1,895491,Hotel Nacional Rio de Janeiro OFICIAL,2.062326,0.789316,NaN,NaN,...,NaN,None,None,None,None,NaN,None,None,None,None
6,1,Quiero un hotel tranquilo en Río de Janeiro co...,alternativa_semantica,2,661634,Wyndham Rio de Janeiro Barra,2.057771,0.787572,NaN,NaN,...,NaN,None,None,None,None,NaN,None,None,None,None
7,1,Quiero un hotel tranquilo en Río de Janeiro co...,alternativa_estructurada,1,231021,Majestic Rio Palace Hotel,NaN,NaN,0.471850,0.471850,...,NaN,None,None,None,None,NaN,None,None,None,None
8,1,Quiero un hotel tranquilo en Río de Janeiro co...,alternativa_estructurada,2,345798,Hotel Atlântico Business Centro,NaN,NaN,0.359249,0.359249,...,NaN,None,None,None,None,NaN,None,None,None,None
9,2,Estoy planeando un viaje a Mar Del Plata. Nece...,principal,1,353326,Hermitage Hotel,1.333810,0.549368,2.000000,1.000000,...,1.0,None,None,None,None,NaN,None,None,None,None


In [37]:
# Mostramos las columnas de revisión de hoteles (relevantes)
columnas_revision_hoteles = [
    "caso_id",
    "tipo_recomendacion",
    "posicion",
    "hotel_name",
    "semantic_score_norm",
    "structured_score_norm",
    "hybrid_score",
    "score_promedio",
    "reviews_afines",
    "vista",
    "desayuno",
    "ubicacion",
    "limpieza",
    "wifi",
    "aire_acondicionado",
    "ejemplos_reviews"
]

df_eval_hoteles[columnas_revision_hoteles]

,caso_id,tipo_recomendacion,posicion,hotel_name,semantic_score_norm,structured_score_norm,hybrid_score,score_promedio,reviews_afines,vista,desayuno,ubicacion,limpieza,wifi,aire_acondicionado,ejemplos_reviews
0,1,principal,1,Royal Rio Palace Hotel,0.821549,1.000000,0.910774,0.836876,12.0,None,1.000000,NaN,None,NaN,NaN,[O hotel é lindo e o café da manhã é incrivel....
1,1,principal,2,Américas Copacabana Hotel,0.850992,0.536193,0.693593,0.842526,13.0,None,0.536193,NaN,None,NaN,NaN,"[El hotel muy lindo cerca de la playa, desayun..."
2,1,principal,3,Laghetto Stilo Barra,0.907994,0.431635,0.669815,0.837356,16.0,None,0.431635,NaN,None,NaN,NaN,"[Muito bem localizado e seguro, otimo café da ..."
3,1,principal,4,Windsor Oceanico,1.000000,0.302949,0.651475,0.845282,21.0,None,0.302949,NaN,None,NaN,NaN,"[Hermoso hotel, frente a la playa, desayuno mu..."
4,1,principal,5,Prodigy Santos Dumont by Wish,0.740616,0.378016,0.559316,0.840396,9.0,None,0.378016,NaN,None,NaN,NaN,[Buenos los desayunos y la tranquilidad en el ...
5,1,alternativa_semantica,1,Hotel Nacional Rio de Janeiro OFICIAL,0.789316,NaN,NaN,0.829941,11.0,None,NaN,NaN,None,NaN,NaN,[O café da manhã é muito bom e o hotel fica na...
6,1,alternativa_semantica,2,Wyndham Rio de Janeiro Barra,0.787572,NaN,NaN,0.828108,11.0,None,NaN,NaN,None,NaN,NaN,[Muy lindo el hotel... Hermosa vista a la play...
7,1,alternativa_estructurada,1,Majestic Rio Palace Hotel,NaN,0.471850,NaN,NaN,NaN,None,0.471850,NaN,None,NaN,NaN,None
8,1,alternativa_estructurada,2,Hotel Atlântico Business Centro,NaN,0.359249,NaN,NaN,NaN,None,0.359249,NaN,None,NaN,NaN,None
9,2,principal,1,Hermitage Hotel,0.549368,1.000000,0.774684,0.685443,6.0,None,0.000000,1.0,None,1.0,NaN,"[Muy lindo hotel y excelente ubicación, Hotel ..."


In [38]:
# Mostramos las columnas de revisión del prompt (relevantes)
columnas_revision_prompt = [
    "caso_id",
    "prompt_usuario",
    "destino_enviado",
    "consulta_semantica_enviada",
    "consulta_atributos_enviada",
    "query_sql",
    "n_recomendaciones_principales",
    "n_alternativas_semanticas",
    "n_alternativas_estructuradas"
]

df_eval_prompts[columnas_revision_prompt]

,caso_id,prompt_usuario,destino_enviado,consulta_semantica_enviada,consulta_atributos_enviada,query_sql,n_recomendaciones_principales,n_alternativas_semanticas,n_alternativas_estructuradas
0,1,Quiero un hotel tranquilo en Río de Janeiro co...,Brasil - Rio De Janeiro,hotel tranquilo con linda vista al mar y rico ...,desayuno,"SELECT hotel_id_review, destino, desayuno FROM...",5,2,2
1,2,Estoy planeando un viaje a Mar Del Plata. Nece...,Argentina - Mar Del Plata,"hotel bien ubicado, con buen desayuno, y funda...","bien ubicado, buen desayuno, fundamental wifi","SELECT hotel_id_review, destino, wifi, ubicaci...",1,2,0
2,3,"Quiero un hotel en Florianopolis, cerca de la ...",Brasil - Florianópolis,hotel cerca de la playa,aire acondicionado,"SELECT hotel_id_review, destino, aire_acondici...",5,2,2


### Conclusiones

A lo largo del módulo construimos un recomendador híbrido conversacional que integra dos motores complementarios desarrollados en los módulos previos, con una capa de re-ranking explícito y un agente conversacional que orquesta la interacción con el usuario.

**Lo que se logró:**

- Integración de los dos motores en una única tool híbrida (`buscar_hoteles_hibrido`) que cruza determinísticamente los rankings semántico y estructurado, en lugar de delegar la combinación a la redacción del LLM. Esto hace que el comportamiento del sistema sea predecible y defendible.

- Separación clara de responsabilidades: el LLM interpreta la consulta del usuario y redacta la respuesta final; los motores recuperan y rankean; la capa híbrida combina las dos señales con una lógica explícita y trazable.

- Cobertura de tres tipos de salida: recomendaciones principales con doble evidencia, alternativas semánticas fuertes en similitud, y alternativas estructuradas fuertes en atributos. Esto permite manejar casos donde la intersección entre listas es chica sin que el sistema se quede sin respuesta.

**Limitaciones conocidas:**

- Ponderación arbitraria del score híbrido: el score que combina las dos señales se calcula como un promedio 50/50 entre el score semántico normalizado y el score estructurado normalizado. Esta proporción se fijó de manera discrecional, sin un análisis de sensibilidad que evaluara cómo cambia la calidad del ranking al variar los pesos (por ejemplo, 70/30 o 30/70). En una iteración futura sería valioso experimentar con distintas proporciones y comparar los resultados contra un conjunto de consultas anotadas, o incluso aprender los pesos a partir de feedback de usuarios.

- El sistema ignora la señal negativa de las reviews: tanto el motor semántico como el estructurado se construyeron tomando únicamente la información positiva. En el módulo 2, los atributos con score negativo se llevan a cero en `score_hotel`, descartando efectivamente las menciones desfavorables al momento de rankear. En el módulo 3 se vectorizó solo el texto positivo de cada review (descartando el texto negativo). Esto simplifica el sistema en su primera versión, pero también desaprovecha información valiosa: un hotel con muchas menciones negativas de wifi probablemente debería penalizarse, no solo "no recibir crédito" por esa categoría. Una mejora natural sería incorporar la señal negativa de forma explícita, tanto en los embeddings (vectorizando también texto negativo) como en el scoring estructurado (usando los sentimientos negativos como factor de penalización).

- Intersección entre listas potencialmente chica: la lógica de recomendaciones principales depende de que un hotel aparezca tanto en el ranking semántico como en el estructurado. Cuando la consulta combina criterios muy distintos (por ejemplo, *"tranquilo cerca del centro con buen wifi"*), esa intersección puede ser pequeña o vacía. El sistema mitiga este caso con las alternativas semánticas y estructuradas, pero la calidad de las recomendaciones principales depende fuertemente de la coherencia entre las dos señales para cada consulta.

- Falta de evaluación cuantitativa: no se realizó una evaluación sistemática del sistema (precision@k, recall@k, satisfacción de usuario) por falta de un conjunto de consultas etiquetadas. Las pruebas del agente son cualitativas y se basan en inspección manual de los outputs.

- Cobertura funcional limitada: el sistema no incorpora información de precio, disponibilidad ni reserva, lo cual lo limita a un rol de "asistente de descubrimiento" antes de la decisión de compra. Tampoco mantiene memoria conversacional entre turnos, por lo que un usuario no puede refinar consultas iterativamente (por ejemplo, *"de esos hoteles, ¿cuál es el más barato?"*).